# HypatiaX Notebook 05: Publication Figure Generation

**Paper:** LLMs as Interfaces to Symbolic Discovery: Perfect Extrapolation via Hybrid Architectures  
**Journal:** Journal of Machine Learning Research (2026)  
**Author:** Dr. Ruperto Pedro Bonet Chaple

This notebook reproduces all publication figures from the JMLR paper.

---

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# ── Repo-root resolution ─────────────────────────────────────────────────────
# Works whether the notebook is run from the repo root, the notebooks/ dir,
# or any subdirectory.  Looks for the marker file hypatiax/data/results.
def _find_repo_root() -> Path:
    """Walk up from this notebook until we find hypatiax/data/results."""
    here = Path().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'results').exists():
            return candidate
        if (candidate / 'hypatiax' / 'data' / 'results').exists():
            return candidate / 'hypatiax'
    raise FileNotFoundError(
        "Cannot locate repo root.  "
        "Run the notebook from inside the LLM-HypatiaX-PAPERS repository."
    )

REPO_ROOT   = _find_repo_root()
RESULTS_DIR = REPO_ROOT / 'data' / 'results'
FIGURES_DIR = REPO_ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Add repo root to sys.path so hypatiax modules are importable
if str(REPO_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT.parent))

print(f"✓ Repo root   : {REPO_ROOT}")
print(f"✓ Results dir : {RESULTS_DIR}")
print(f"✓ Figures dir : {FIGURES_DIR}")

MAIN_FILE    = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_131438.json'
MAIN_FILE2   = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_133510.json'

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.neural_network import MLPRegressor

# Publication style
plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'font.family': 'serif',
    'figure.dpi': 150
})

# Create output directory

# Load data
with open(MAIN_FILE) as f:
    raw = json.load(f)
df = pd.DataFrame(raw)

df['equation_name'] = df['metadata'].apply(lambda x: x.get('equation_name', ''))
df['difficulty']    = df['metadata'].apply(lambda x: x.get('difficulty', ''))
df['formula_type']  = df['metadata'].apply(lambda x: x.get('formula_type', ''))
df['llm_r2']        = df['llm_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['nn_r2']         = df['nn_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['eval_r2']       = df['evaluation'].apply(lambda x: x.get('r2', None) if x else None)
df['decision']      = df['evaluation'].apply(lambda x: x.get('decision', 'llm') if x else 'llm')
df['decision_reason']= df['evaluation'].apply(lambda x: x.get('decision_reason', '') if x else '')
df['decision']       = df['evaluation'].apply(lambda x: x.get('decision', 'llm') if x else 'llm')
df['decision_reason']= df['evaluation'].apply(lambda x: x.get('decision_reason', '') if x else '')
df['success']        = df['eval_r2'].apply(lambda x: x >= 0.90 if x is not None else False)

print(f'✓ Setup complete — {len(df)} equations loaded')
print(f'✓ Figures directory: {FIGURES_DIR.absolute()}')

## 2. Figure 1: Arrhenius Catastrophic Failure

In [ ]:
np.random.seed(42)
A, Ea, R_c = 1e13, 50000, 8.314

T_train  = np.linspace(300, 400, 200)
k_train  = A * np.exp(-Ea / (R_c * T_train)) * (1 + np.random.normal(0, 0.01, 200))
T_extrap = np.linspace(200, 600, 500)
k_true   = A * np.exp(-Ea / (R_c * T_extrap))

nn = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=3000, random_state=42)
nn.fit(T_train.reshape(-1,1), np.log(k_train))
k_nn  = np.exp(nn.predict(T_extrap.reshape(-1,1)))
k_sym = A * np.exp(-Ea / (R_c * T_extrap))

sym_err = np.abs(k_sym - k_true) / k_true
nn_err  = np.abs(k_nn  - k_true) / k_true

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

ax1.semilogy(T_extrap, k_true, 'k-',  lw=2.5, label='True: $k = Ae^{-E_a/RT}$', zorder=3)
ax1.semilogy(T_extrap, k_sym,  'g--', lw=2,   label='HypatiaX (Symbolic)', zorder=2)
ax1.semilogy(T_extrap, k_nn,   'r:',  lw=2,   label='Neural Network', zorder=1)
ax1.axvspan(300, 400, alpha=0.15, color='steelblue', label='Training Range')
ax1.set_xlabel('Temperature T (K)')
ax1.set_ylabel('Rate Constant k (log scale)')
ax1.set_title('(a) Predictions: Training vs Extrapolation')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(T_extrap, sym_err + 1e-16, 'g-', lw=2, label=f'HypatiaX  (median: {np.median(sym_err):.1e})')
ax2.semilogy(T_extrap, nn_err  + 1e-16, 'r-', lw=2, label=f'Neural Net (median: {np.median(nn_err):.1e})')
ax2.axvspan(300, 400, alpha=0.15, color='steelblue')
ax2.axhline(y=1e-12, color='gray', ls='--', lw=1.5, label='Float. point precision (10⁻¹²)')
ax2.set_xlabel('Temperature T (K)')
ax2.set_ylabel('Relative Error')
ax2.set_title('(b) Extrapolation Error Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')

plt.suptitle('Figure 1: Arrhenius Equation — Catastrophic Neural Network Failure\nvs Near-Perfect Symbolic Extrapolation',
             fontsize=12, fontweight='bold')
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIGURES_DIR / f'figure1_arrhenius_extrapolation.{ext}', dpi=300, bbox_inches='tight')
print('✓ Saved Figure 1')
plt.show()

## 3. Figure 2: Domain Success Rate Comparison

In [ ]:
valid = df[df['eval_r2'].notna()].copy()
domain_stats = valid.groupby('domain').agg(
    n=('eval_r2', 'count'),
    success=('success', 'sum'),
    mean_r2=('eval_r2', 'mean')
)
domain_stats['success_rate'] = domain_stats['success'] / domain_stats['n'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# (a) Success rate
ax = axes[0]
colors = sns.color_palette('husl', len(domain_stats))
bars = ax.bar(domain_stats.index, domain_stats['success_rate'], color=colors, alpha=0.85, edgecolor='white')
ax.axhline(y=domain_stats['success_rate'].mean(), color='red', ls='--', lw=2,
           label=f'Overall: {domain_stats["success_rate"].mean():.1f}%')
ax.set_ylabel('Success Rate (%)')
ax.set_xlabel('Domain')
ax.set_title('(a) Success Rate by Domain')
ax.set_ylim([0, 115])
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for bar, (_, row) in zip(bars, domain_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1.5,
            f'{row["success_rate"]:.0f}%\n(n={int(row["n"])})', ha='center', fontsize=9)

# (b) R² distribution by domain
ax = axes[1]
for i, (domain, group) in enumerate(valid.groupby('domain')):
    r2_vals = group['eval_r2'].clip(-1, 1)
    ax.scatter([i] * len(r2_vals), r2_vals, alpha=0.6, s=60,
               color=colors[i], label=domain.capitalize())
    ax.plot([i-0.3, i+0.3], [r2_vals.mean(), r2_vals.mean()],
            color=colors[i], lw=3, solid_capstyle='round')
ax.axhline(y=0.90, color='red', ls='--', lw=1.5, label='Success threshold (0.90)')
ax.set_xticks(range(len(domain_stats)))
ax.set_xticklabels([d.capitalize() for d in domain_stats.index], rotation=45)
ax.set_ylabel('R² Score')
ax.set_title('(b) R² Score Distribution by Domain')
ax.legend(bbox_to_anchor=(1.05, 1))
ax.grid(True, alpha=0.3)

plt.suptitle('Figure 2: Success Rate and R² Across Benchmark Domains',
             fontsize=12, fontweight='bold')
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIGURES_DIR / f'figure2_domain_comparison.{ext}', dpi=300, bbox_inches='tight')
print('✓ Saved Figure 2')
plt.show()

## 4. Figure 3: LLM vs NN Decision Routing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Decision distribution
ax = axes[0]
decision_counts = df['decision'].value_counts()
wedge_colors = ['#2ecc71', '#e74c3c', '#3498db']
wedges, texts, autotexts = ax.pie(
    decision_counts.values,
    labels=[f'{d.upper()}\n({v})' for d, v in decision_counts.items()],
    autopct='%1.1f%%',
    colors=wedge_colors[:len(decision_counts)],
    startangle=90,
    pctdistance=0.7
)
ax.set_title('(a) Decision Routing\n(LLM vs NN)')

# (b) LLM R² vs NN R² scatter
ax = axes[1]
valid_b = df[df['llm_r2'].notna() & df['nn_r2'].notna()].copy()
llm_r2c = valid_b['llm_r2'].clip(-2, 1)
nn_r2c  = valid_b['nn_r2'].clip(-2, 1)

llm_wins = (llm_r2c > nn_r2c)
ax.scatter(nn_r2c[llm_wins],  llm_r2c[llm_wins],  color='#2ecc71', alpha=0.8, s=70, label='LLM wins')
ax.scatter(nn_r2c[~llm_wins], llm_r2c[~llm_wins], color='#e74c3c', alpha=0.8, s=70, label='NN wins', marker='^')
ax.plot([-2, 1], [-2, 1], 'k--', alpha=0.4)
ax.set_xlabel('Neural Network R²')
ax.set_ylabel('LLM R²')
ax.set_title('(b) LLM vs NN R²\n(all equations)')
ax.legend()
ax.grid(True, alpha=0.3)

# (c) Decision by domain
ax = axes[2]
decision_by_domain = df.groupby(['domain', 'decision']).size().unstack(fill_value=0)
decision_by_domain.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'][:len(decision_by_domain.columns)],
                         alpha=0.85, edgecolor='white')
ax.set_title('(c) Decision Routing by Domain')
ax.set_xlabel('Domain')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Decision')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Figure 3: LLM vs Neural Network Decision Routing', fontsize=12, fontweight='bold')
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIGURES_DIR / f'figure3_decision_routing.{ext}', dpi=300, bbox_inches='tight')
print('✓ Saved Figure 3')
plt.show()

## 5. Figure 4: Performance by Formula Type and Difficulty

In [ ]:
valid = df[df['eval_r2'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (a) By formula type
ax = axes[0]
type_stats = valid.groupby('formula_type').agg(
    n=('eval_r2', 'count'),
    success_rate=('success', lambda x: x.mean() * 100)
).sort_values('success_rate', ascending=True)

colors_t = ['#e74c3c' if s < 80 else '#f39c12' if s < 95 else '#2ecc71'
            for s in type_stats['success_rate']]
bars = ax.barh(type_stats.index, type_stats['success_rate'], color=colors_t, alpha=0.85)
ax.axvline(x=95.8, color='red', ls='--', lw=2, label='Paper avg: 95.8%')
ax.set_xlabel('Success Rate (%)')
ax.set_title('(a) Success Rate by Formula Type')
ax.set_xlim([0, 115])
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
for bar, (_, row) in zip(bars, type_stats.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2.,
            f'{row["success_rate"]:.0f}% (n={int(row["n"])})', va='center', fontsize=9)

# (b) R² heatmap by domain × difficulty
ax = axes[1]
pivot = valid.pivot_table(values='eval_r2', index='domain', columns='difficulty', aggfunc='mean')
# Reorder columns
col_order = [c for c in ['easy', 'medium', 'hard'] if c in pivot.columns]
pivot = pivot[col_order]

sns.heatmap(pivot, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Mean R²'})
ax.set_title('(b) Mean R² by Domain × Difficulty')
ax.set_xlabel('Difficulty')
ax.set_ylabel('Domain')

plt.suptitle('Figure 4: Performance by Formula Type and Difficulty', fontsize=12, fontweight='bold')
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIGURES_DIR / f'figure4_formula_difficulty.{ext}', dpi=300, bbox_inches='tight')
print('✓ Saved Figure 4')
plt.show()

## 6. Figure 5: Method Comparison Summary

In [ ]:
# System comparison — computed from loaded data
valid_fig5 = df[df['eval_r2'].notna()].copy()
valid_fig5['success_f5'] = valid_fig5['eval_r2'] >= 0.90

hybrid_sr = valid_fig5['success_f5'].mean() * 100
hybrid_r2 = valid_fig5['eval_r2'].mean()

llm_valid  = valid_fig5[valid_fig5['llm_r2'].notna()]
llm_sr     = (llm_valid['llm_r2'] >= 0.95).mean() * 100
llm_r2_m   = llm_valid['llm_r2'].clip(-1, 1).mean()

nn_valid   = valid_fig5[valid_fig5['nn_r2'].notna()]
nn_sr      = (nn_valid['nn_r2'] >= 0.90).mean() * 100
nn_r2_m    = nn_valid['nn_r2'].clip(-1, 1).mean()

# Runtime estimates from paper Table (seconds, median per equation)
RUNTIME_LLM, RUNTIME_HYBRID, RUNTIME_NN = 15, 390, 120

systems       = ['Pure LLM', 'Hybrid (HypatiaX)', 'Neural Network']
success_rates = [llm_sr,      hybrid_sr,           nn_sr]
mean_times    = [RUNTIME_LLM, RUNTIME_HYBRID,      RUNTIME_NN]
mean_r2       = [llm_r2_m,   hybrid_r2,            nn_r2_m]

print(f'Computed stats — LLM: {llm_sr:.1f}%  Hybrid: {hybrid_sr:.1f}%  NN: {nn_sr:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c']

# (a) Success rates
ax = axes[0]
bars = ax.bar(systems, success_rates, color=colors, alpha=0.85, edgecolor='white', width=0.5)
ax.axhline(y=95, color='gray', ls='--', lw=1.5, alpha=0.7, label='Target: 95%')
ax.set_ylabel('Success Rate (%)')
ax.set_title('(a) Success Rate')
ax.set_ylim([0, 115])
ax.tick_params(axis='x', rotation=15)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, success_rates):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold')

# (b) Mean R²
ax = axes[1]
bars = ax.bar(systems, mean_r2, color=colors, alpha=0.85, edgecolor='white', width=0.5)
ax.set_ylabel('Mean R²')
ax.set_title('(b) Mean R² Score')
ax.set_ylim([0.5, 1.05])
ax.tick_params(axis='x', rotation=15)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, mean_r2):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontweight='bold')

# (c) Success vs Speed tradeoff
ax = axes[2]
scatter = ax.scatter(mean_times, success_rates, s=[400, 600, 300],
                     c=colors, alpha=0.8, edgecolors='black', linewidth=1.5, zorder=3)
for i, sys in enumerate(systems):
    ax.annotate(sys, (mean_times[i], success_rates[i]),
                xytext=(15, 5), textcoords='offset points',
                fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax.set_xlabel('Mean Discovery Time (seconds, log scale)')
ax.set_ylabel('Success Rate (%)')
ax.set_title('(c) Success-Speed Tradeoff')
ax.set_xscale('log')
ax.axhline(y=95, color='green', ls='--', lw=1.5, alpha=0.6, label='Target: 95%')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Figure 5: Method Comparison — Pure LLM vs Hybrid vs Neural Network',
             fontsize=12, fontweight='bold')
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIGURES_DIR / f'figure5_method_comparison.{ext}', dpi=300, bbox_inches='tight')
print('✓ Saved Figure 5')
plt.show()

## 7. Figure Summary

In [ ]:
figures = list(FIGURES_DIR.glob('figure*.pdf'))
print(f'Generated {len(figures)} publication figures:')
for f in sorted(figures):
    size_kb = f.stat().st_size / 1024
    print(f'  ✓ {f.name} ({size_kb:.1f} KB)')

---
**Previous:** [04_extrapolation_analysis.ipynb](04_extrapolation_analysis.ipynb)  
**Next:** [06_statistical_tests.ipynb](06_statistical_tests.ipynb)